# Corridas com posição de origem e destino (Silver)

As solicitações não trazem coordenadas. Para corridas encerradas, a posição é obtida do **GPS do próprio motorista** nos marcos da corrida: aceite (`ts_aceite`), embarque (`ts_inicio`, proxy da origem) e desembarque (`ts_fim`, proxy do destino).

## Fluxo dos dados

* Origens: `tb_solicitacoes`, `tb_posicoes_motoristas` e `tb_deslocamentos_motoristas` da Silver
* Destino: `cat_mobilidade_urbana_silver.db_frota.tb_corridas`

## Regras

1. Só entram solicitações `ATENDIDA`. Pedidos não atendidos não têm posição de origem nesta fonte; a demanda espacial é, portanto, **demanda atendida**, e a cobertura deve ser reportada como condicional (matriz de regras, §19 item 4).
2. Para cada marco, usa-se a posição válida do mesmo motorista com carimbo GPS mais próximo, até 180 s de distância. `gap_*_seg` guarda a diferença assinada. A posição é **confiável** com |gap| ≤ 120 s.
3. `km_*_reta`: Haversine entre os marcos (linha reta). `km_gps_*`: soma dos trechos GPS válidos associados à solicitação no estado `A_CAMINHO` (pickup) ou `COM_PASSAGEIRO` (viagem). Ambos são estimativas, nunca odômetro (RN008, RN012).

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

## 1. Marcos das corridas encerradas

In [0]:
corridas_base = spark.table(f"{SILVER}.tb_solicitacoes").where("categoria_desfecho = 'ATENDIDA'")

eventos = (
    corridas_base.select(
        "id_solicitacao",
        "id_motorista",
        F.explode(
            F.array(
                F.struct(F.lit("aceite").alias("evento"), F.col("ts_aceite").alias("ts_evento")),
                F.struct(F.lit("origem").alias("evento"), F.col("ts_inicio").alias("ts_evento")),
                F.struct(F.lit("destino").alias("evento"), F.col("ts_fim").alias("ts_evento")),
            )
        ).alias("e"),
    )
    .select("id_solicitacao", "id_motorista", "e.evento", "e.ts_evento")
    .where(F.col("ts_evento").isNotNull() & F.col("id_motorista").isNotNull())
    .withColumn("_minuto", F.floor(F.col("ts_evento").cast("double") / 60))
    .withColumn("bucket", F.explode(F.sequence(F.col("_minuto") - 3, F.col("_minuto") + 3)))
    .drop("_minuto")
)

## 2. Posição mais próxima de cada marco

A junção usa baldes de 1 minuto do mesmo motorista para evitar uma junção por intervalo sobre 10 milhões de posições.

In [0]:
posicoes = (
    spark.table(f"{SILVER}.tb_posicoes_motoristas")
    .where("fl_coordenada_valida AND fl_posicao_atual")
    .select("id_motorista", "ts_posicao", "latitude", "longitude", "h3_r8", "h3_r7")
    .withColumn("bucket", F.floor(F.col("ts_posicao").cast("double") / 60))
)

candidatas = (
    eventos.join(posicoes, ["id_motorista", "bucket"])
    .withColumn("gap_seg", segundos_entre("ts_evento", "ts_posicao"))
    .where(F.abs("gap_seg") <= 180)
)
mais_proxima = (
    candidatas.withColumn(
        "_rn",
        F.row_number().over(
            W.partitionBy("id_solicitacao", "evento").orderBy(F.abs("gap_seg"), "ts_posicao")
        ),
    )
    .where("_rn = 1")
    .drop("_rn", "bucket")
)


def campo(evento, coluna, nome):
    return F.first(F.when(F.col("evento") == evento, F.col(coluna)), ignorenulls=True).alias(nome)


posicoes_marcos = mais_proxima.groupBy("id_solicitacao").agg(
    *[
        c
        for ev in ("aceite", "origem", "destino")
        for c in (
            campo(ev, "latitude", f"lat_{ev}"),
            campo(ev, "longitude", f"lon_{ev}"),
            campo(ev, "h3_r8", f"h3_r8_{ev}"),
            campo(ev, "h3_r7", f"h3_r7_{ev}"),
            campo(ev, "gap_seg", f"gap_{ev}_seg"),
        )
    ]
)

## 3. Quilometragem por GPS amostrado

In [0]:
gps = (
    spark.table(f"{SILVER}.tb_deslocamentos_motoristas")
    .where(F.col("id_solicitacao_inicio").isNotNull())
    .groupBy(F.col("id_solicitacao_inicio").alias("id_solicitacao"), "id_motorista")
    .agg(
        F.sum(F.when(F.col("fl_passo_valido") & (F.col("categoria_km") == "A_CAMINHO"), F.col("km_haversine"))).alias("km_gps_pickup"),
        F.sum(F.when(F.col("fl_passo_valido") & (F.col("categoria_km") == "COM_PASSAGEIRO"), F.col("km_haversine"))).alias("km_gps_viagem"),
        F.sum((F.col("categoria_km") == "A_CAMINHO").cast("int")).alias("qtd_trechos_pickup"),
        F.sum((F.col("categoria_km") == "COM_PASSAGEIRO").cast("int")).alias("qtd_trechos_viagem"),
        F.sum((~F.col("fl_passo_valido")).cast("int")).alias("qtd_trechos_invalidos"),
    )
)

## 4. Montagem e persistência

In [0]:
corridas = (
    corridas_base.select(
        "id_solicitacao", "id_regiao", "tipo_solicitacao", "id_motorista", "id_veiculo",
        "ts_registro", "ts_aceite", "ts_chegada", "ts_inicio", "ts_fim",
        "data_local", "hora_local", "dia_semana_iso", "fl_fim_de_semana",
        "seg_registro_ate_aceite", "seg_pickup", "seg_espera_embarque", "seg_viagem",
        "fl_tempos_ordenados", "fl_aguardou_condutor_em_outra_viagem",
    )
    .join(posicoes_marcos, "id_solicitacao", "left")
    .join(gps, ["id_solicitacao", "id_motorista"], "left")
    .withColumn("km_pickup_reta", haversine_km("lat_aceite", "lon_aceite", "lat_origem", "lon_origem"))
    .withColumn("km_viagem_reta", haversine_km("lat_origem", "lon_origem", "lat_destino", "lon_destino"))
    .withColumn("fl_aceite_confiavel", F.coalesce(F.abs("gap_aceite_seg") <= 120, F.lit(False)))
    .withColumn("fl_origem_confiavel", F.coalesce(F.abs("gap_origem_seg") <= 120, F.lit(False)))
    .withColumn("fl_destino_confiavel", F.coalesce(F.abs("gap_destino_seg") <= 120, F.lit(False)))
    .withColumn(
        "fl_corrida_espacial_valida",
        F.col("fl_tempos_ordenados") & F.col("fl_origem_confiavel") & F.col("fl_destino_confiavel"),
    )
    .withColumn("metodo_posicao", F.lit("gps_motorista_mais_proximo_do_marco"))
    .withColumn("_ts_processamento", F.current_timestamp())
)

gravar(
    corridas,
    f"{SILVER}.tb_corridas",
    "Corridas encerradas com origem, destino e posicao de aceite inferidas pelo GPS do motorista no marco mais proximo "
    "(ate 180 s; confiavel ate 120 s). KM reta e KM por GPS amostrado sao estimativas.",
)

In [0]:
for linha in (
    spark.table(f"{SILVER}.tb_corridas")
    .agg(
        F.count("*").alias("corridas"),
        F.avg(F.col("fl_origem_confiavel").cast("int")).alias("pct_origem"),
        F.avg(F.col("fl_destino_confiavel").cast("int")).alias("pct_destino"),
        F.avg(F.col("fl_aceite_confiavel").cast("int")).alias("pct_aceite"),
        F.avg(F.col("fl_corrida_espacial_valida").cast("int")).alias("pct_valida"),
        F.expr("percentile_approx(km_pickup_reta, array(0.5, 0.9))").alias("p_km_pickup_reta"),
        F.expr("percentile_approx(km_gps_pickup, array(0.5, 0.9))").alias("p_km_gps_pickup"),
        F.expr("percentile_approx(km_viagem_reta, array(0.5, 0.9))").alias("p_km_viagem_reta"),
    )
    .collect()
):
    print(linha.asDict())